# 물류센터 SLA 개선 시뮬레이션 — 전체 정리

B2C 풀필먼트센터의 **정시 출고(SLA) 위반 문제**를 이산사건(대기행렬) 시뮬레이션으로 재현하고,
개선안들을 **정직하게** 검증한 기록입니다. 아이디어가 "그럴듯하다"에서 멈추지 않고, 실제로
효과가 있는지(혹은 역효과인지)를 데이터로 판정하는 것이 목표입니다.

## 한눈에 보는 결론

| 개선안 | 유형 | 결과 |
|---|---|---|
| 3-1 보충 시점 재분배 | 부하 **재배치** | ✗ 무효/역효과 |
| 3-2 슬로팅 이동 | 부하 **재배치** | ✗ 역효과 |
| 3-3 저녁 배수 용량 | 용량 **추가** | ○ 효과(유료, +9~10% 노동) |
| **NEW-1 마감 우선순위(트리아지)** | 용량 **재배분** | ○ **효과(무비용, 위반 −37%)** |
| 근본: 보충 강도↓ | 부하 **축소** | ○ (설비 투자/수요 평준화로만 실현) |

> **핵심 통찰:** 시스템이 **~99% 포화**라서, 부하를 *시간·공간으로 옮기는*(3-1·3-2) 것만으로는
> SLA를 못 바꾼다. 효과는 ①용량 추가(3-3), ②용량 재배분(NEW-1), ③부하 축소(근본)에서만 나온다.
> **실행 우선순위: NEW-1(즉시·무비용) → 3-3(전환기·유료) → 근본 레버(중장기).**

**권고 패키지(§6):** 무비용 트리아지에 정밀 타겟 저녁 용량을 얹으면 **위반 22.7→8.4~5.8%
(−63~74%), 포커스 37.4→12.7~16.9%**를 달성하고, 저녁 용량이 트리아지의 지각 꼬리까지 치료한다.


## 0. 배경과 접근

**문제:** 화~목요일 14~17시에 인입된 주문의 SLA 위반율이 나머지 구간의 약 2배
(포커스 37.4% vs 베이스 19.0%, 전체 23.0%).

**근본 메커니즘(진단):** 프로모션이 화~목에 집중 → 대응 보충작업이 오후에 배치로 몰림 →
지게차·통로 간섭으로 오후 피킹 처리량 ~20% 저하 → 유입은 일정한데 처리량만 떨어져 **백로그
누적** → 주말에만 배수되어 **주중(화→금) 이월** → 오후 인입 주문이 익일 데드라인을 넘김.

**시뮬레이션 접근:** 이 인과 사슬을 하나의 **공유 FIFO 대기행렬**로 축약한다.
- 도착: 주문라인이 `allocated_time`에 픽 큐 진입(실데이터)
- 서버: 시간대별 활성 작업자 `W(t)`가 라인당 `s_eff`만큼의 작업자-초 소비
- 서비스시간: `s_eff(c) = base_service + [pick(c) − pick(0)]`
  ( `pick(c)` = 그 라인 zone의 **동시 보충 c건**일 때 픽 서비스시간 = **간섭 비용함수**, 실측 )

자유 파라미터는 `base_service` **하나뿐**(전체 SLA 23%에 맞춰 보정), 나머지는 전부 데이터에서 추출.
아래 셀들은 `sim/` 패키지의 파이프라인을 그대로 호출한다.


In [ ]:
# --- 환경 설정 & 데이터·파라미터 로드 ---
import sys, os, pickle, subprocess
import pandas as pd

SIM = r'C:\coding\haksuljae\sim'
sys.path.insert(0, SIM)
os.chdir(SIM)                       # outputs/ 상대경로 · 캐시 접근용
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 25)

import io_load, engine, policies, scenarios
from config import OUT_DIR, TARGETS
from IPython.display import Image, display

lines  = io_load.load_lines()       # 주문라인 1개 = 1행 (시뮬 기본 입력)
repl   = io_load.load_repl()        # 보충작업 로그
master = io_load.load_master()      # SKU 마스터(zone·거리·회전율)

params_path = os.path.join(OUT_DIR, 'calib_params.pkl')
if not os.path.exists(params_path):
    import calibrate; params = calibrate.run_calibration()
else:
    params = pickle.load(open(params_path, 'rb'))

# 그림이 없으면 전체 파이프라인으로 (재)생성
if not os.path.exists(os.path.join(OUT_DIR, 'fig_discipline.png')):
    subprocess.run([sys.executable, 'run.py', '--scenarios'], cwd=SIM, check=True)

print(f'주문라인 {len(lines):,}건 · 보충 {len(repl):,}건 · SKU {len(master)}종 로드 완료')
print(f'baseline 재현 목표(실측): 전체 {TARGETS["sla_overall"]}% / '
      f'포커스 {TARGETS["sla_focus"]}% / 베이스 {TARGETS["sla_base"]}%')

**위 출력의 의미:** 6개월치 주문라인 39.7만 건과 보충 2.2만 건을 로드했다. 아래 모든 분석은
이 실데이터와, 데이터에서 추출한 캘리브레이션 파라미터(`calib_params.pkl`) 위에서 돈다.


## 1. 캘리브레이션 & baseline 재현

개선안을 검증하려면 먼저 시뮬레이터가 **현실(baseline)을 재현**해야 한다.

특히 **간섭 비용함수**(동시 보충 c → 픽 서비스시간)의 *형태*가 결정적이다. 멘토(이상수님)가
지적했듯 "타 팀은 유사 개선안에서 효과없음/역효과가 나왔으니, 비용함수 형태(볼록 vs 오목)를
먼저 검증하라". 아래에서 그 형태를 데이터로 확인한다.


In [ ]:
# 간섭 비용함수 형태 (데이터에서 적합)
c = params['cost']
print('동시 보충 c → 픽 서비스시간(초):')
for k in ['0','1-5','6-20','21-50','50+']:
    print(f'  버킷 {k:>5}: {c["bucket_mean"][k]:.0f}s')
print(f'형태 = {"오목(포화)" if c["concave"] else "기타"} · '
      f'첫 계단(0→1~5)이 전체 팽창의 {c["first_step_share"]*100:.0f}%')
display(Image('outputs/fig_cost_function.png'))

**결과 — 비용함수는 극단적으로 오목(포화):** 0→1건에서 +189초(전체 팽창의 62%)로 폭증하고
그 뒤는 거의 평평하다. 즉 "보충이 **있느냐/없느냐**"의 이진에 가깝고, "몇 건이냐"는 부차적이다.

이 형태는 3-1·3-2에 대한 **역효과 복선**이다 — 계단형 비용함수에서는 간섭을 시간·공간으로
옮겨도(펼쳐도) *간섭받는 픽의 수가 줄지 않아* 총 간섭이 보존되기 때문이다(뒤에서 확인).


In [ ]:
# baseline 시뮬 실행 → 실측 재현 확인
res = engine.simulate(lines, params, policies.baseline(repl))   # discipline='fifo'
s = engine.summarize(res)
print(f"시뮬 baseline : 전체 {s['sla_overall']:.1f}% / 포커스 {s['sla_focus']:.1f}% / 베이스 {s['sla_base']:.1f}%")
print(f"목표(실측)    : 전체 {TARGETS['sla_overall']}  / 포커스 {TARGETS['sla_focus']}  / 베이스 {TARGETS['sla_base']}")
display(Image('outputs/fig_sla_heatmap.png'))
display(Image('outputs/fig_backlog_sawtooth.png'))

**결과 — 재현 성공:** 시뮬이 실측 SLA(포커스 37.4% / 베이스 19%)를 거의 정확히 맞춘다.
자유 파라미터는 `base_service` 하나뿐인데도:

- **요일×시간 히트맵**(위): 월요일 0% · 화~목 오후 상승 · 금요일 종일 높음이 관측과 일치.
- **주간 백로그 톱니**(아래): 화→금 누적, 주말 배수 패턴이 **캘리브레이션에 넣지 않았는데도
  창발**한다. 특히 *금요일 종일 지연*(당일 보충이 적은데도)은 이미 높은 진입 백로그가 이월된
  결과로, 백로그 이월 메커니즘을 독립적으로 검증한다.

→ 이 baseline 위에서 개선안을 검증하면 신뢰할 수 있다.


## 2. 개선안 3-1 — 보충 시점 재분배 (부하 재배치)

**아이디어(제출물):** 오후 14~17시에 집중된 비긴급 보충(threshold·promo_prepare, 91.7%)을
오전 유휴 시간대로 옮겨 오후 간섭을 줄인다. 설비 증설 없이 스케줄만 조정.

**모델링:** 두 방식을 비교한다.
- `s31_shift`: 오후 배치를 **통째로 오전으로 평행이동**(타이밍만 이동)
- `s31_spread`: 오전에 **얇게 펼침**(문서의 '평준화' 문구 그대로)

비용함수가 오목이므로 **역효과**가 예상된다(위 §1 복선). 아래에서 확인한다.


In [ ]:
# 3-1 정책 시뮬 (baseline 대비)
scen = scenarios.run_policies(lines, repl, params)
b = scen['baseline']['summary']
rows = {}
for nm, lab in [('baseline','baseline'),('s31_shift','3-1 시점이동'),('s31_spread','3-1 평준화')]:
    m = scen[nm]['summary']
    rows[lab] = {'전체 SLA%': round(m['sla_overall'],1), '포커스 SLA%': round(m['sla_focus'],1),
                 'Δ포커스': round(m['sla_focus']-b['sla_focus'],1),
                 '총 간섭(픽시간h)': round(m['total_excess_h'])}
display(pd.DataFrame(rows).T)
display(Image('outputs/fig_scenario_concurrency.png'))

**결과 — 3-1은 효과 없음/역효과:** 포커스 SLA가 37.4%에서 오히려 38~39%로 **소폭 악화**된다.
그림에서 보듯 3-1은 간섭의 **봉우리를 오후→오전으로 옮길 뿐**, 총 간섭 픽시간은 8,297h로 거의
보존된다(오목 비용함수의 필연적 귀결 — 간섭받는 픽 수가 그대로).

게다가 시스템이 ~99% 포화라 오전으로 옮긴 간섭이 **촉박한 당일 데드라인(408분) 픽까지 지연**시켜
손해다. 아래 형태 민감도가 "왜 오목이면 안 되는가"를 못박는다.


In [ ]:
# 비용함수 '형태' 민감도 — 볼록이면 펼칠 때 이득, 오목/선형은 아님 (정책=평준화)
sens = scenarios.cost_shape_sensitivity(lines, repl, params)
display(sens[['shape','baseline_focus','s31_spread_focus','delta_focus','delta_excess_h']].round(1))

**결과 — 형태가 부호를 결정한다:** 보충을 펼쳤을 때 총 간섭 변화(`delta_excess_h`)가
**볼록이면 −697h(감소·이득)**, 선형이면 +947h(증가), **오목(실측)이면 +17h(≈0)**이다.
즉 재배치가 유효하려면 비용함수가 볼록이어야 하는데, **데이터는 오목**이라 이득이 사실상 0.
"형태를 먼저 검증하라"는 멘토 지적이 정확했음을 정량 확인한다.


## 3. 개선안 3-2 — 슬로팅 재최적화 (부하 재배치)

**아이디어(제출물):** 과밀한 ZB·ZD(적재율 123%)의 고회전 SKU 일부를 유휴 ZE·ZF(19%)로
재배치해 건당 보충 소요를 67분→32분으로 낮추고 간섭을 줄인다.

**핵심 상충(제 프록시가 놓쳤던 것):** ZB는 스테이징에 **가깝고(거리 17)** 고회전, ZE/ZF는
**멀다(거리 74)**. 고회전 SKU를 먼 곳으로 옮기면 **픽 이동거리가 4.4배**로 뛴다
(실측: clean 픽시간 ≈ 2.46초 × 거리). 이 거리 페널티를 포함해 정식으로 검증한다.


In [ ]:
# 3-2 판정: '순수 강도감소(이상적 상한)' vs '실현 슬로팅(거리 페널티 포함)'
verdict = scenarios.slotting_verdict(lines, repl, master, params)
display(verdict.round(1))
display(Image('outputs/fig_slotting.png'))

**결과 — 3-2도 역효과:** ZB/ZD 보충 강도만 낮추면(이동 없이) 포커스 37.4→35.4%로 개선되지만,
이는 거리 페널티가 0인 **물리적으로 실현 불가능한 상한**이다. 그 강도 감소를 실제 수단(SKU 이동)
으로 실현하는 순간, 고회전 SKU의 픽 거리가 늘어 용량을 깎고 SLA가 **41.8%로 악화**된다.

결정적으로, 낙관적 가정(남은 ZB/ZD도 빨라짐)으로 **총 간섭이 baseline보다 낮아져도(6,840<8,297)
SLA는 여전히 악화**된다 — ~99% 포화에서 거리 페널티에 의한 용량 손실이 지배적이기 때문이다.


## 4. 개선안 3-3 — 저녁 배수 용량 (용량 추가)

**아이디어(제출물):** 3-1·3-2와 달리 부하를 옮기는 게 아니라, 진입 백로그가 높은 요일의
**저녁 시간대에 처리 용량을 한시적으로 보강**해 다음날로 넘어가는 이월분을 소진한다.

**모델링:** 저녁(19~23시)에 추가 인원을 투입(엔진 `worker_boost`). 이건 포화를 직접 완화하므로
효과가 있을 것으로 예상된다 — 관건은 **필요 최소 용량과 초과근무 비용**이다. 요일 타겟팅
(화~목 저녁)이 전면 적용보다 얼마나 효율적인지 본다.


In [ ]:
# 3-3: 저녁 추가 인원 스윕 (요일 타겟 = 화~목 저녁 19~23시)
fr = scenarios.evening_frontier(lines, repl, params)
tgt = fr['Tue-Thu×19-23'][['n_extra','sla_focus','sla_overall','overtime_h','ot_pct_labor']]
display(tgt.round(1).rename(columns={'n_extra':'추가인원','sla_focus':'포커스SLA%',
        'sla_overall':'전체SLA%','overtime_h':'초과근무h','ot_pct_labor':'노동증가%'}))
display(Image('outputs/fig_evening.png'))

**결과 — 3-3은 효과 있으나 유료:** 저녁 용량을 추가하니 포커스 SLA가 실제로 낮아진다.
**화~목 저녁 +20명이면 포커스 37.4→18.9%(베이스 수준)** 를 초과근무 +9.4% 노동으로 달성한다.

왼쪽 그림(비용-효익 프론티어)에서 **요일 타겟(화~목)이 Mon-Fri 전면 적용보다 같은 SLA를 40%
적은 초과근무로** 달성함을 보인다(문서의 '필요 최소 용량' 취지). 오른쪽은 저녁 보강이 주중
백로그 톱니를 눌러 이월을 끊는 모습이다. 단, **상시 초과근무 비용이 드는 반응형 대책**이다.


## 5. 신규 개선안 NEW-1 — 마감 기반 우선순위 피킹 (용량 재배분, 무비용)

3-1·3-2(부하 재배치)가 실패하고 3-3(용량 추가)이 유료임을 확인했다. 남은 미탐색 영역은
**기존 용량을 마감 기준으로 더 똑똑하게 배분**하는 것 — 용량·부하를 그대로 두고 **무비용**으로
SLA를 개선할 수 있는가?

**모델링(엔진 `discipline`):**
- `edd`: 마감 빠른 순으로 처리
- `triage`: **이미 늦은(구제 불가) 주문을 후순위로 미뤄**, 아직 구제 가능한 주문에 용량을 집중.
  단 무한 지연을 막는 **지연 상한(cap)** 을 둔다.

**논리:** 현행은 사실상 FIFO(도착순). 포화 큐는 이미 늦은 주문에 용량을 낭비하는데, 이를 트리아지
하면 같은 용량으로 위반 '건수'를 줄일 수 있다(포화 큐의 지각작업 최소화).


In [ ]:
# NEW-1: 큐 규율 비교 (FIFO / EDD / 트리아지 cap별)
dm = scenarios.run_disciplines(lines, repl, params)
view = dm[['sla_overall','sla_focus','late_max_h','over_1d_pct','over_2d_pct']].copy()
view['late_max_h'] = (view['late_max_h']/24).round(1)          # 최대 지각(일)
view = view.rename(columns={'sla_overall':'전체%','sla_focus':'포커스%',
        'late_max_h':'최대지각(일)','over_1d_pct':'>1일%','over_2d_pct':'>2일%'})
display(view.round(1))
display(Image('outputs/fig_discipline.png'))

**결과 — 무비용으로 작동하는 첫 레버:**

1. **순수 EDD(마감우선)는 FIFO와 동일** — 포화 큐에선 "마감 빠른 순 = 가장 늦은 것부터"라 효과 無.
2. **트리아지가 위반 건수를 크게 줄인다** — 이미 늦은 주문에 용량 낭비를 멈추고 구제 가능한
   주문을 살리기 때문. 단 doomed 주문의 지각 폭이 커지는 **상충**이 있다(cap=∞면 최대 7.5일).
3. **지연 상한(cap)으로 꼬리를 통제** — **cap≈12h면 위반 22.7→14.2%(−37%)** 에 최대 지각을
   33시간으로 억제하고 2일 초과는 0%.

→ **권고: cap≈12h 균형 트리아지.** 소프트웨어/정책 변경만으로 위반을 37% 줄인다. 단, SLA를
'위반 건수'로 볼 때 유효하며(꼬리 지각은 감수), 지각 '심도'를 중시하면 cap을 더 낮춘다.


## 6. 권고 패키지 — 레버 스태킹 (해커톤 클라이맥스)

개별 레버를 넘어, **효과가 검증된 것들을 쌓는다.** 3-1·3-2(부하 재배치)는 버리고:

- **① 트리아지 (무비용):** 큐 규율만 바꿔 위반을 먼저 줄인다.
- **② + 정밀 타겟 저녁 용량 (저비용):** 화~목 저녁에만 최소 인원을 얹어 백로그를 배수한다.

"공짜 최적화를 먼저 → 필요한 **최소** 용량만"이라는 순서가 핵심이다(무작정 증원이 아님).
아래에서 레버가 곱셈적으로 쌓이는지, 그리고 저녁 용량이 트리아지의 약점(지각 꼬리)을
치료하는지 확인한다.


In [ ]:
# 권고 패키지: 트리아지(무비용) → 저녁 용량(저비용) 스태킹
combo = scenarios.run_combo(lines, repl, params)
view = combo.copy()
view = view.rename(columns={'label':'단계','cost':'비용','sla_overall':'전체%',
        'sla_focus':'포커스%','late_max_d':'최대지각(일)','over_1d_pct':'>1일%',
        'overtime_pct':'초과근무%'}).set_index('단계')
display(view.round(1))
display(Image('outputs/fig_combo.png'))

**결과 — 이것이 권고안:** 레버가 곱셈적으로 쌓인다.

| 단계 | 전체 위반 | 포커스 | 비용 |
|---|---|---|---|
| 현행 | 22.7% | 37.4% | — |
| ① 트리아지 | 14.2% | 30.7% | **무비용** |
| ② +저녁 20명 | **8.4%** (−63%) | **16.9%** | +9.4% 노동 |
| ② +저녁 30명 | **5.8%** (−74%) | **12.7%** | +14.2% 노동 |

- **큰 효과:** 위반을 현행의 **1/4 수준**으로, 포커스 SLA를 **절반 이하**로.
- **★시너지(오른쪽 그림):** 트리아지 단독의 약점인 '>1일 지각 5.6%'가, 저녁 용량을 더하면
  0.4~1.3%로 **치료**된다 — 저녁 배수로 doomed 주문 자체가 줄기 때문. 두 레버가 서로의
  단점을 메운다.
- **비용 효율:** 무작정 증원(포화만큼 다 채우기)이 아니라, 무비용 최적화로 부하를 먼저 줄인
  뒤 남는 만큼만 최소 용량으로 메운다.

→ **권고: 트리아지 + 화~목 저녁 20명** (위반 −63%, 초과근무 +9.4%, 지각 꼬리 없음). 더 공격적
목표면 +30명(−74%).


## 7. 종합 결론

| 개선안 | 유형 | 결과 | 비용 |
|---|---|---|---|
| 3-1 시점 재분배 | 부하 재배치 | ✗ 무효/역효과 | — |
| 3-2 슬로팅 이동 | 부하 재배치 | ✗ 역효과 | — |
| 3-3 저녁 용량 | 용량 추가 | ○ 효과 | 초과근무 +9~10% 노동(상시) |
| **NEW-1 트리아지** | 용량 재배분 | ○ **효과(위반 −37%)** | **무비용**(꼬리 지각 상충) |
| 근본: 보충 강도↓ | 부하 축소 | ○ (실현은 자본/협업) | 설비 투자 or 수요 평준화 |

**왜 이런 결과인가 — 시스템이 ~99% 포화이기 때문이다.** 포화 시스템에서 부하를 시간·공간으로
'재배치'(3-1·3-2)하는 것은 총량을 못 바꾸므로 SLA를 못 바꾼다. 효과는 오직 ①용량 추가(3-3),
②용량 재배분(NEW-1), ③부하 축소(근본)에서만 나온다.

**실행 우선순위:** **NEW-1(즉시·무비용) → 3-3(전환기·유료) → 근본 레버(중장기)**.

> **본선 스토리:** "① 제안했던 3-1·3-2를 시뮬로 검증했더니 **역효과**였고, 그 과정에서 진짜
> 병목이 **포화 용량**임을 규명했다. ② 그래서 **무비용 트리아지로 먼저 −37%**, ③ 여기에 정밀
> 타겟 최소 용량을 얹어 **위반 −63~74%·포커스 절반**을 달성하고 지각 꼬리까지 제거했다."
> — 아이디어를 검증해 방향을 뒤집고, 검증된 레버를 쌓아 큰 효과를 낸 것이 스토리다.

*(재현: `sim/` 폴더에서 `python run.py --scenarios`. 상세는 `sim/README.md`.)*
